# Planner SFT v3 — EXAONE-3.5-7.8B LoRA 파인튜닝 실증 보고
## teacher 증류 → LoRA SFT → holdout A/B

**작성일**: 2026-07-09 · **대상**: `mongle-ai` planner(일정 분할) 노드 · **관련**: `sft_pipeline/experiments/planner_sft_v3/`, ADR-0005

> 본 노트북은 한 번의 파인튜닝 사이클(증류·학습·평가·A/B)을 논문 형식으로 기록한다. 모든 수치는 실제 실행에서 관측한 값이며, 추정은 명시한다. 목적은 **재현 가능성**이다 — 동일 절차·동일 환경 핀으로 다시 돌릴 수 있도록 환경 함정까지 남긴다.

## 초록 (Abstract)

몽글마을 planner 노드는 자연어 목표를 날짜별 실행 계획(JSON: `summary_text`/`days`)으로 분할한다. 베이스 EXAONE-3.5-7.8B-Instruct는 이 스키마를 안정적으로 지키지 못했다(holdout 파싱 성공률 10%). 

이를 개선하기 위해 (1) GPT-4o teacher로 960개 합성 입력을 증류하고 의미 판정·구조 검증 필터로 621건을 채택, (2) EXAONE-3.5-7.8B에 LoRA(r=16)를 1 epoch SFT(최종 train_loss 0.20), (3) 동일 holdout 30건에서 base와 LoRA를 A/B 비교했다.

결과: **JSON 스키마 준수율(파싱 성공률) 10% → 93%**, 의미 점수(1–5) 3.5 → 3.82, 시험 정보·외국어 누출 0건. 승격 게이트(게이트 통과 AND 의미평균 LoRA ≥ base)를 통과했다. 본 보고는 결과와 함께 **EXAONE × transformers 5.x 환경 함정 6종**과 그 우회를 재현 가능하게 정리한다.

## 실행·재현 노트

- 코드 데이터 셀은 **외부 의존성 없이** 실행된다(표준 라이브러리만). 차트 셀은 `matplotlib`이 있으면 그래프를, 없으면 텍스트 막대를 출력한다.
- 수치는 2026-07-09 단발 실행이다. holdout 30건, 단일 시드·단일 런이므로 §5 한계를 함께 본다.
- 학습은 RunPod GPU(RTX 4090, 24GB, ephemeral)에서, 증류는 CPU/API(로컬)에서 수행했다.

## 1. 서론 및 문제 정의

### 1.1 대상 노드
planner는 `{plan_kind, goal_text, today, deadline, slots}` 입력을 받아 `{summary_text, days:[{date, tasks:[{title, due_date}]}], personalization_patch}` 를 출력한다. 날짜 정확도는 코드가 재계산하므로, 모델에는 **순서·구조·내용 품질**이 요구된다.

### 1.2 문제
베이스 EXAONE-3.5-7.8B-Instruct는 프롬프트만으로는 이 계약을 지키지 못했다. holdout 30건 중 스키마를 만족하는 파싱 가능 출력은 3건(10%)에 그쳤다(§3). 즉 **형식 신뢰도**가 서비스 적용의 병목이었다.

### 1.3 왜 EXAONE인가
planner는 한국어 일정·시험 도메인을 다룬다. 텍스트 기능 전반은 벤치마크로 선정한 Qwen2.5-7B를 쓰되(별도 보고), planner 노드만 한국어 특화 EXAONE-3.5-7.8B를 전용 SFT 대상으로 분리했다. 목표는 **파인튜닝으로 형식 신뢰도를 끌어올릴 수 있음의 증명**이다.

## 2. 방법

```
① 증류(distill)      GPT-4o teacher + 의미판정/구조검증 필터 → gold 621건
② 학습(SFT)          EXAONE-3.5-7.8B + LoRA(r=16), 1 epoch
③ 평가(holdout)      LoRA 30건 생성 → 규칙 게이트 + 의미 판정
④ A/B                동일 holdout 을 base 로도 생성 → base vs LoRA 비교
```
A/B와 승격은 **train==serve 계약**(같은 스키마·같은 파서)으로 판정한다.

### 2.1 데이터 증류
goal_corpus가 도메인(event·exam·lifestyle·project·routine) 균형으로 **960개 입력**을 생성한다. 각 입력을 GPT-4o teacher가 계획으로 변환하고, **구조 검증**(days 비어있음·task 개수 초과 등)과 **의미 판정**(1–5, 임계 미만 DROP)으로 걸러 gold를 만든다. holdout 30건은 고정(5) + 분포 미러(25).

In [ ]:
# 증류 결과 (2026-07-09 로컬 재실행, GPT-4o teacher)
distill = {
    'corpus_inputs': 960,
    'processed': 935,
    'accepted': 621,
    'holdout': 30,
    'accept_rate': round(621/935, 3),
}
print(f"채택 {distill['accepted']}/{distill['processed']} (accept_rate={distill['accept_rate']})")
print(f"holdout {distill['holdout']}건 · gold {distill['accepted']}건")
# 참고: accept_rate ~66%. 코퍼스 960 기준 700 바닥에 근접하나 미달 → 학습 바닥값을 500으로 낮춰 진행(§5).

### 2.2 학습 설정
EXAONE는 unsloth 미지원이라 **표준 transformers Trainer + peft**로 학습한다. 4bit(QLoRA)는 현재 bitsandbytes와 비호환이라 자동으로 **bf16 폴백**(EXAONE 7.8B bf16 ≈ 16GB, 24GB에 적재). responses-only 손실(프롬프트 접두사 토큰을 -100 마스킹).

In [ ]:
train = {
    'base_model': 'LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct',
    'method': 'LoRA (r=16, alpha=16, dropout=0.0)',
    'precision': 'bf16 (4bit 폴백)',
    'train_rows': 558, 'valid_rows': 63,
    'epochs': 1, 'lr': 2e-4, 'batch': 1, 'grad_accum': 8,
    'steps': 70, 'wall_clock_min': 8, 'gpu': 'RTX 4090 (24GB)',
    'final_train_loss': 0.2026,
}
# loss 곡선(로그 기록)
loss_curve = [0.2767, 0.2156, 0.1873, 0.1886, 0.1801, 0.1901, 0.1790]
for k, v in train.items():
    print(f"{k:16}: {v}")
print('loss:', ' -> '.join(str(x) for x in loss_curve))

### 2.3 평가·A/B 프로토콜
holdout 30건을 (a) LoRA, (b) base(어댑터 없음)로 각각 생성한다. 각 출력에 대해:
- **규칙 게이트**: 파싱 성공률, 구조 위반율, 마감 준수율, 시험 정보 누출, 외국어 누출
- **의미 판정**: GPT-4o judge가 계획-목표 정합성을 1–5로 채점

승격 자격 = 게이트 통과 **AND** 의미평균(LoRA) ≥ 의미평균(base).

## 3. 결과 (base vs LoRA, 동일 holdout 30건)

In [ ]:
# eval_report.json / eval_report_base.json 에서 관측한 지표
base = {'parse_rate': 0.10, 'structure_violation_rate': 0.00, 'deadline_rate': 1.00,
        'exam_leak': 0, 'english_leak': 0, 'semantic_avg': 3.5, 'passed': False}
lora = {'parse_rate': 0.9333, 'structure_violation_rate': 0.0357, 'deadline_rate': 0.9643,
        'exam_leak': 0, 'english_leak': 0, 'semantic_avg': 3.82, 'passed': True}

print(f"{'metric':26} {'base':>8} {'LoRA':>8}")
for k in ['parse_rate', 'structure_violation_rate', 'deadline_rate', 'semantic_avg']:
    print(f"{k:26} {base[k]:>8.3f} {lora[k]:>8.3f}")
print(f"{'passed':26} {str(base['passed']):>8} {str(lora['passed']):>8}")
print()
print('핵심: parse_rate(스키마 준수율) 0.10 -> 0.93')

In [ ]:
def show_bars(title, labels, values):
    """matplotlib 있으면 막대, 없으면 텍스트 폴백."""
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(6, 0.6*len(labels)+1))
        ax.barh(range(len(labels)), values)
        ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
        ax.invert_yaxis(); ax.set_title(title)
        for i, v in enumerate(values): ax.text(v, i, f' {v}', va='center')
        plt.tight_layout(); plt.show()
    except ModuleNotFoundError:
        mx = max(values) or 1
        print(title)
        for l, v in zip(labels, values):
            print(f'  {l:14.14} | {chr(9608)*int(30*v/mx)} {v}')

show_bars('파싱 성공률 (스키마 준수율)', ['base', 'LoRA'], [base['parse_rate'], lora['parse_rate']])
print()
show_bars('의미 점수 (1-5)', ['base', 'LoRA'], [base['semantic_avg'], lora['semantic_avg']])

### 3.1 해석
- **parse_rate 0.10 → 0.93 이 주된 성과.** 베이스는 요구 스키마대로 된 계획을 거의 못 내고(3/30), SFT 후 28/30이 정상 생성된다. 파인튜닝이 **형식 신뢰도**를 극적으로 올렸음을 보인다.
- **semantic_avg 3.5 → 3.82 는 소폭.** 베이스도 파싱만 되면 내용 품질은 나쁘지 않다. LoRA의 이득은 품질보다 **신뢰성·구조**에 있다.
- base의 `structure_violation 0.00`은 **3건만 파싱돼 생긴 소표본 착시**이므로 단독 해석하지 않는다.
- 시험 정보·외국어 누출은 양쪽 0건.

## 4. 환경 함정·재현 (EXAONE × transformers 5.x)

> 이 절이 재현의 핵심이다. EXAONE-3.5의 커스텀 modeling 코드는 특정 transformers 버전을 전제하며, 최신(5.9+)과는 여러 지점에서 어긋난다. 아래를 그대로 적용하면 학습→평가가 통과한다.

| # | 증상 | 원인 | 우회 |
|---|---|---|---|
| 1 | `create_causal_mask() unexpected kwarg 'input_embeds'` | 최신 transformers가 시그니처 변경 | **`transformers==5.5.0` 핀** (근본 해결 — 아래 여러 개가 이걸로 사라짐) |
| 2 | `OverflowError: ... Encoding` (datasets.map) | `apply_chat_template(tokenize=True)`가 `List[int]` 아닌 Encoding 반환 | `apply_chat_template(tokenize=False)`로 문자열 → `tok(text, add_special_tokens=False)['input_ids']` |
| 3 | `get_input_embeddings not auto-handled for ExaoneModel` (학습) | EXAONE가 임베딩 미노출 | `get_peft_model` 전에 첫 `nn.Embedding`을 찾아 `model.get_input_embeddings` 수동 노출 |
| 4 | 동일 에러 (평가 `_load`) | `hasattr` 체크가 raise를 못 걸러냄 | `try/except (NotImplementedError, AttributeError)`로 감싸 노출 |
| 5 | 학습 후 `CUDA OOM` → 어댑터 미저장 | `eval_strategy='epoch'` in-loop 평가가 저장 전에 OOM | `eval_strategy='no'` (저장이 평가보다 먼저) + `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` |
| 6 | `parse_rate 0.0` (평가) | 모델이 ```json``` 펜스로 감싸 출력, 파서가 `json.loads` 직행 | 파서가 `{`~`}` 부분문자열만 추출 후 로드 |
| + | `pip`이 torch 2.10(915MB) 재다운로드 | requirements의 `unsloth`가 최신 torch를 끌어옴(EXAONE는 unsloth 미사용) | 학습 requirements에서 `unsloth`·`trl` 제외 |
| + | `ModuleNotFoundError: openai` (채점) | judge가 OpenAI 사용, 학습 requirements엔 없음 | 평가 전 `pip install openai` |

패치 위치: 2·3·5 → `sft_pipeline/train/train_plain.py`, 4·6 → `sft_pipeline/experiments/planner_sft_v3/{evaluate.py,contract.py}`.

## 5. 논의 (한계)

- **암기 가능성.** 최종 train_loss 0.20은 암기 바닥(0.3) 아래다. 558건 1 epoch(각 샘플 ~1회 노출)이라 순수 암기로 보긴 어렵고, 구조화 JSON은 원래 저엔트로피라 loss가 낮게 나오는 면이 있다. 다만 **holdout(안 본 데이터)에서 parse_rate 0.93**로 일반화가 확인되므로 형식 학습은 과적합이 아니다. 내용 품질(semantic 3.82)의 일반화는 holdout 30건 규모 안에서만 주장한다.
- **소표본.** holdout 30건, 단일 시드·단일 런. 신뢰구간·분산 미측정. 운영 전 대표본 재평가를 권한다.
- **데이터 규모.** accept_rate ~66%로 gold 621건(목표 700 미만). 바닥값을 낮춰 진행했으며, teacher 드롭 사유(의미 저점·구조 위반)를 줄이면 규모·품질을 함께 올릴 수 있다.
- **judge 편향.** teacher와 judge가 모두 GPT-4o 계열이라 판정이 teacher 문체에 관대할 수 있다. 규칙 게이트(파싱·누출)는 이 편향에서 독립적이다.

## 6. 결론

- EXAONE-3.5-7.8B planner 노드를 LoRA로 SFT하여 **일정 분할 스키마 준수율을 10% → 93%로 개선**하고 승격 게이트를 통과했다. 파인튜닝으로 형식 신뢰도를 끌어올릴 수 있음을 실측으로 증명했다.
- 재현의 관건은 **`transformers==5.5.0` 핀 + §4의 우회 6종**이다.
- 다음: (1) holdout 확대·다중 시드로 분산 측정, (2) teacher 드롭 축소로 gold 규모↑, (3) 승격 시 운영 `LORA_PLANNER_REPO` 교체(A/B 게이트 통과 후에만).

## 부록 A. 재현 명령 (RunPod 24GB)

```bash
# 0) 데이터(gold/holdout)는 git 미추적 → HF private dataset 경유
hf download bigmooon/planner-sft-v3-data --repo-type dataset \
  --local-dir sft_pipeline/experiments/planner_sft_v3/data

# 1) 앱 의존성 + 버전 핀 (핵심)
pip install -r requirements-api.txt        # pydantic·openai·langgraph (torch 미포함)
pip install 'transformers==5.5.0'          # EXAONE 호환 핀
# 학습 requirements 에서 unsloth·trl 제외(주석)

# 2) 학습 (§4 패치 적용된 train_plain.py, 바닥값 500)
EXPERIMENT_ROOT=outputs/planner-sft-v3-run \
  bash sft_pipeline/experiments/planner_sft_v3/train_runpod.sh

# 3) 평가(LoRA) / A/B(base)
export OPENAI_API_KEY=...
for A in outputs/planner-sft-v3-run/adapter base; do
  PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
  python -m sft_pipeline.experiments.planner_sft_v3.evaluate \
    --adapter $A --base-model LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct \
    --holdout sft_pipeline/experiments/planner_sft_v3/data/holdout.jsonl \
    --out outputs/planner-sft-v3-run/eval_${A##*/}.json
done
```

**환경 요약**: RTX 4090(24GB) · Python 3.11 · torch 2.4(RunPod 베이스) · transformers 5.5.0 · peft·datasets·bitsandbytes(bf16 폴백) · teacher/judge GPT-4o.

**산출물(HF, private)**: 어댑터 `bigmooon/exaone-planner-sft-v3-run1` · 데이터 `bigmooon/planner-sft-v3-data`.